# Task 3.2 — Gold: Instructor Dashboard
## Notebook: 09_gold_instructor_dashboard

**Layer:** Gold (weekly instructor-level aggregates)

**What this notebook does:**
- Aggregates per instructor from Gold and Silver sources:
    → gold.course_performance_daily    : active_courses, total_active_students,
                                         avg_completion_rate, avg_quiz_pass_rate
    → silver.enrollment_risk_profile   : students_at_high_dropout_risk
    → bronze.course_catalog_bronze     : instructor metadata (name, course list)
- Computes performance_rank: RANK() OVER (ORDER BY avg_completion_rate DESC)
- Computes completion_rate_4w_avg: avg of completion_rate over last 4 weekly snapshots
  (current week vs 4-week rolling average for trend context)
- Writes to gold.instructor_dashboard — weekly Monday refresh

**Source tables:**
    mini_project_grp6.bronze.course_catalog_bronze
    mini_project_grp6.gold.course_performance_daily
    mini_project_grp6.silver.enrollment_risk_profile

**Target table:** mini_project_grp6.gold.instructor_dashboard

**Business Rules enforced:**
- Every active instructor (has at least one active course) must appear
- performance_rank must be unique (no ties — use RANK not DENSE_RANK per spec)
- avg_completion_rate must be between 0.0 and 1.0
- students_at_high_dropout_risk must be >= 0
- week_start_date must not be NULL

**Run order:** Run all cells top-to-bottom. Safe to re-run (replaceWhere on week_start_date).

In [0]:
# ============================================================
# CELL 2 — Catalog config and all path/table name constants
# Run this cell first every time before executing any other cell
# ============================================================

# --- Catalog & schema names ---
CATALOG = "mini_project_grp6"
BRONZE  = "bronze"
SILVER  = "silver"
GOLD    = "gold"

# --- Source tables ---
CATALOG_BRONZE_TABLE    = f"{CATALOG}.{BRONZE}.course_catalog_bronze"
COURSE_PERF_GOLD_TABLE  = f"{CATALOG}.{GOLD}.course_performance_daily"
RISK_PROFILE_TABLE      = f"{CATALOG}.{SILVER}.enrollment_risk_profile"

# --- Target table ---
INSTRUCTOR_DASH_TABLE   = f"{CATALOG}.{GOLD}.instructor_dashboard"
DQ_AUDIT_TABLE          = f"{CATALOG}.audit.dq_audit"

# --- Week configuration ---
# In production: Airflow passes week_start_date = last Monday
# For sample data: derive from the latest report_date in Gold
# week_start_date = Monday of the week containing max(report_date)

# --- 4-week average lookback ---
LOOKBACK_WEEKS = 4

# --- Schema version tag ---
SCHEMA_VERSION = "v1.0"

# --- Set default catalog + schema ---
spark.catalog.setCurrentCatalog(CATALOG)
spark.catalog.setCurrentDatabase(GOLD)

print(f"✅ Catalog : {CATALOG}")
print(f"✅ Schema  : {GOLD}")
print(f"✅ Target  : {INSTRUCTOR_DASH_TABLE}")
print()
print("Sources:")
print(f"   {CATALOG_BRONZE_TABLE}")
print(f"   {COURSE_PERF_GOLD_TABLE}")
print(f"   {RISK_PROFILE_TABLE}")

In [0]:
# ============================================================
# CELL 3 — All imports needed for this notebook
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StringType, IntegerType,
    DoubleType, DateType
)
from delta.tables import DeltaTable

print("✅ Imports successful")

## Part A — Determine week_start_date

The instructor dashboard refreshes weekly on Monday.
week_start_date is always the Monday of the current week.

In production: Airflow passes this as a DAG parameter.
For sample data: derive from the latest report_date available
in gold.course_performance_daily by rolling back to the
nearest Monday using date_trunc('WEEK', max_report_date).

The 4-week average compares:
  current week  : week_start_date
  prior 3 weeks : week_start_date - 7d, -14d, -21d

In [0]:
# ============================================================
# CELL 5 — Determine week_start_date
#
# In production: Airflow passes WEEK_START_DATE as a job parameter
# For sample data: we derive from max(report_date) in Gold table
#
# date_trunc('WEEK', date) → returns the Monday of that week
# in Spark (ISO week standard — week starts Monday)
# ============================================================

gold_df = spark.table(COURSE_PERF_GOLD_TABLE)

# Get the latest report_date available in Gold
max_report_date = gold_df.agg(F.max("report_date")).collect()[0][0]

# Derive the Monday of the week containing max_report_date
week_start_row = spark.sql(
    f"SELECT date_trunc('WEEK', DATE '{max_report_date}') AS week_start"
).collect()[0]

WEEK_START_DATE = week_start_row["week_start"]

# 4-week lookback start: 4 weeks before week_start_date
LOOKBACK_START_DATE = spark.sql(
    f"SELECT date_sub(DATE '{WEEK_START_DATE}', {LOOKBACK_WEEKS * 7}) AS lb_start"
).collect()[0]["lb_start"]

print(f"✅ max_report_date in Gold  : {max_report_date}")
print(f"✅ week_start_date (Monday) : {WEEK_START_DATE}")
print(f"✅ 4-week lookback start    : {LOOKBACK_START_DATE}")
print(f"ℹ  In production Airflow passes WEEK_START_DATE directly")

## Part B — Build Instructor → Course Mapping

Source: bronze.course_catalog_bronze

The catalog gives us:
- instructor_id and instructor_name per course_id
- is_active flag to know if the course is currently running

We join this to Gold course performance data so every metric
can be rolled up per instructor.

Only courses with is_active = 1 are included (inactive/retired
courses should not count against an instructor's live metrics).

In [0]:
# ============================================================
# CELL 7 — Build instructor to course mapping
#
# We need instructor_id and instructor_name per course_id.
# Filter to is_active = 1 so inactive courses don't pollute
# the instructor's live performance metrics.
#
# If a course has is_active = 0, it still has historical data
# in Gold but we exclude it from the instructor's current dashboard.
# ============================================================

catalog_df = spark.table(CATALOG_BRONZE_TABLE)

# Get active courses with their instructor
instructor_course_map = (
    catalog_df
    .filter(F.col("is_active") == 1)
    .select(
        "course_id",
        "instructor_id",
        "instructor_name",
        "subject_area",
        "difficulty_level",
    )
    .dropDuplicates(["course_id"])   # one row per course
)

active_courses = instructor_course_map.count()
active_instructors = instructor_course_map.select("instructor_id").distinct().count()

print("── Instructor course mapping ─────────────────────")
print(f"   Active courses    : {active_courses:,}")
print(f"   Active instructors: {active_instructors:,}")
print()

print("── Courses per instructor stats ──────────────────")
instructor_course_map.groupBy("instructor_id") \
    .agg(F.count("course_id").alias("num_courses")) \
    .select(
        F.min("num_courses").alias("min_courses"),
        F.max("num_courses").alias("max_courses"),
        F.round(F.avg("num_courses"), 2).alias("avg_courses")
    ).show()

## Part C — Course-Level Metrics from Gold

Pull the latest course performance metrics from gold.course_performance_daily.

For the weekly dashboard we use the most recent report_date available
per course_id (the latest snapshot). In production with daily partitions
this would always be yesterday's data.

Metrics pulled per course:
- active_students
- completion_rate
- course_quiz_pass_rate  (avg student satisfaction proxy)
- avg_engagement_score
- dropout_count
- instructor_response_rate

In [0]:
# ============================================================
# CELL 9 — Pull latest course metrics from gold.course_performance_daily
#
# We want the most recent snapshot per course_id.
# Use ROW_NUMBER() OVER (PARTITION BY course_id ORDER BY report_date DESC)
# to get rank=1 = the latest row per course.
#
# This handles cases where the Gold table has multiple report_dates
# (full historical table). We only want the latest for the dashboard.
# ============================================================

gold_df = spark.table(COURSE_PERF_GOLD_TABLE)

# Window to rank rows per course by most recent report_date
latest_window = Window.partitionBy("course_id").orderBy(F.desc("report_date"))

latest_course_metrics = (
    gold_df
    .withColumn("row_rank", F.row_number().over(latest_window))
    .filter(F.col("row_rank") == 1)
    .select(
        "course_id",
        "report_date",
        "active_students",
        "completion_rate",
        "course_quiz_pass_rate",
        "avg_engagement_score",
        "dropout_count",
        "instructor_response_rate",
    )
    .drop("row_rank")
)

print("── Latest course metrics pulled ──────────────────")
print(f"   Distinct courses: {latest_course_metrics.count():,}")
print()
print("── Stats ─────────────────────────────────────────")
latest_course_metrics.select(
    F.round(F.avg("active_students"), 2).alias("avg_active"),
    F.round(F.avg("completion_rate"), 4).alias("avg_completion"),
    F.round(F.avg("course_quiz_pass_rate"), 4).alias("avg_quiz_pass"),
    F.round(F.avg("avg_engagement_score"), 2).alias("avg_engagement")
).show()

## Part D — High Dropout Risk Students per Instructor

Source: silver.enrollment_risk_profile

For each instructor, count the number of their students currently
classified as HIGH dropout risk across all their active courses.

This is the most actionable metric for instructors —
it tells them exactly how many students need immediate attention.

We join enrollment_risk_profile → course_catalog_bronze on course_id
to attribute risk counts back to the instructor level.

In [0]:
# ============================================================
# CELL 11 — Count students_at_high_dropout_risk per instructor
#
# Join risk profile (course_id grain) to instructor mapping
# to aggregate HIGH risk counts at instructor level.
#
# Only ACTIVE enrollments with HIGH tier are counted.
# COMPLETED/DROPPED/PAUSED are excluded — they are not actionable
# for the instructor at this point.
# ============================================================

risk_df = spark.table(RISK_PROFILE_TABLE)

high_risk_per_course = (
    risk_df
    .filter(
        (F.col("dropout_risk_tier") == "HIGH") &
        (F.col("enrollment_status") == "ACTIVE")
    )
    .groupBy("course_id")
    .agg(
        F.countDistinct("student_id").alias("high_risk_students")
    )
)

# Join to instructor mapping to roll up to instructor level
high_risk_per_instructor = (
    instructor_course_map
    .join(high_risk_per_course, on="course_id", how="left")
    .fillna({"high_risk_students": 0})
    .groupBy("instructor_id")
    .agg(
        F.sum("high_risk_students").alias("students_at_high_dropout_risk")
    )
)

total_high_risk = high_risk_per_instructor \
    .agg(F.sum("students_at_high_dropout_risk")).collect()[0][0]

print("── HIGH risk students per instructor ─────────────")
print(f"   Total HIGH risk students (all instructors): {total_high_risk:,}")
print()

print("── Distribution of high_risk_students per instructor ──")
high_risk_per_instructor.select(
    F.min("students_at_high_dropout_risk").alias("min"),
    F.max("students_at_high_dropout_risk").alias("max"),
    F.round(F.avg("students_at_high_dropout_risk"), 2).alias("avg")
).show()

print("── Top 5 instructors with most HIGH risk students ──")
high_risk_per_instructor.orderBy(F.desc("students_at_high_dropout_risk")).show(5)

## Part E — Aggregate All Metrics per Instructor

Join instructor_course_map → latest_course_metrics on course_id,
then GROUP BY instructor_id to produce one row per instructor.

Aggregations:
- active_courses          : COUNT(course_id) where active_students > 0
- total_active_students   : SUM(active_students)
- avg_completion_rate     : AVG(completion_rate) across instructor's courses
- avg_quiz_pass_rate      : AVG(course_quiz_pass_rate) — satisfaction proxy
- avg_engagement_score    : AVG(avg_engagement_score) across courses

In [0]:
# ============================================================
# CELL 13 — Join course metrics to instructor mapping
#           and aggregate per instructor_id
# ============================================================

# Join course metrics onto the instructor-course mapping
instructor_course_df = (
    instructor_course_map
    .join(latest_course_metrics, on="course_id", how="left")
    .fillna({
        "active_students":      0,
        "completion_rate":      0.0,
        "course_quiz_pass_rate": 0.0,
        "avg_engagement_score": 0.0,
        "dropout_count":        0,
    })
)

# Aggregate to instructor level
instructor_metrics_df = (
    instructor_course_df
    .groupBy("instructor_id", "instructor_name")
    .agg(
        # Active courses = courses where at least 1 student is actively enrolled
        F.sum(
            F.when(F.col("active_students") > 0, 1).otherwise(0)
        ).alias("active_courses"),
        # Total students actively enrolled across all their courses
        F.sum("active_students").alias("total_active_students"),
        # Average completion rate across all their courses
        F.round(F.avg("completion_rate"), 4).alias("avg_completion_rate"),
        # avg quiz pass rate = student satisfaction proxy
        F.round(F.avg("course_quiz_pass_rate"), 4).alias("avg_quiz_pass_rate"),
        # avg engagement score across their courses
        F.round(F.avg("avg_engagement_score"), 2).alias("avg_engagement_score"),
        # Total dropouts across their courses
        F.sum("dropout_count").alias("total_dropout_count"),
    )
)

print("── Instructor metrics aggregated ─────────────────")
print(f"   Total instructors: {instructor_metrics_df.count():,}")
print()
print("── Stats ─────────────────────────────────────────")
instructor_metrics_df.select(
    F.round(F.avg("active_courses"), 2).alias("avg_active_courses"),
    F.round(F.avg("total_active_students"), 2).alias("avg_total_students"),
    F.round(F.avg("avg_completion_rate"), 4).alias("overall_avg_completion"),
    F.round(F.avg("avg_quiz_pass_rate"), 4).alias("overall_avg_quiz_pass")
).show()

In [0]:
# ============================================================
# CELL 14 — Compute performance_rank
#
# RANK() OVER (ORDER BY avg_completion_rate DESC)
#
# RANK() (not DENSE_RANK) per spec:
#   If two instructors share the same avg_completion_rate,
#   they get the same rank and the next rank is skipped.
#   e.g. 1, 2, 2, 4 (not 1, 2, 2, 3)
#
# ORDER BY avg_completion_rate DESC → rank 1 = best instructor
# Secondary sort by avg_quiz_pass_rate DESC to break ties deterministically
# ============================================================

rank_window = Window.orderBy(
    F.desc("avg_completion_rate"),
    F.desc("avg_quiz_pass_rate")   # tiebreaker
)

instructor_ranked_df = (
    instructor_metrics_df
    .withColumn(
        "performance_rank",
        F.rank().over(rank_window)
    )
)

print("── performance_rank distribution ─────────────────")
print(f"   Total ranked instructors: {instructor_ranked_df.count():,}")
print()

print("── Top 10 instructors by rank ────────────────────")
instructor_ranked_df.orderBy("performance_rank").select(
    "performance_rank",
    "instructor_id",
    "instructor_name",
    "active_courses",
    "total_active_students",
    "avg_completion_rate",
    "avg_quiz_pass_rate"
).show(10, truncate=False)

print("── Bottom 5 instructors by rank ──────────────────")
instructor_ranked_df.orderBy(F.desc("performance_rank")).select(
    "performance_rank",
    "instructor_id",
    "instructor_name",
    "avg_completion_rate",
    "total_active_students"
).show(5, truncate=False)

## Part F — 4-Week Rolling Average for Historical Trend

The dashboard spec requires: current week vs 4-week average.

Approach:
- If gold.instructor_dashboard has prior weekly snapshots (prior runs),
  compute AVG(avg_completion_rate) over the last 4 week_start_date entries
  per instructor from the existing table.
- If this is the first run (no prior data), completion_rate_4w_avg = current week value.
- week_over_week_delta = current avg_completion_rate - completion_rate_4w_avg
  (positive = improving, negative = declining vs 4-week trend)

In [0]:
# ============================================================
# CELL 16 — Compute 4-week rolling average of avg_completion_rate
#
# Reads prior snapshots from the existing instructor_dashboard Gold table.
# If the table doesn't exist or has fewer than 4 weeks of data,
# we gracefully fall back to using the current week's value as the avg.
#
# This logic runs BEFORE writing the new snapshot so the 4w avg
# reflects the prior 4 weeks (not including today's new data).
# ============================================================

table_exists = spark.catalog.tableExists(INSTRUCTOR_DASH_TABLE)

if table_exists:
    prior_snapshots = spark.table(INSTRUCTOR_DASH_TABLE)
    prior_weeks_count = prior_snapshots.select("week_start_date") \
                                        .distinct().count()

    print(f"ℹ  Prior weekly snapshots found: {prior_weeks_count}")

    if prior_weeks_count >= 1:
        # Compute avg completion_rate over the last LOOKBACK_WEEKS snapshots per instructor
        lookback_window = (
            Window
            .partitionBy("instructor_id")
            .orderBy(F.desc("week_start_date"))
            .rowsBetween(0, LOOKBACK_WEEKS - 1)
        )

        four_week_avg_df = (
            prior_snapshots
            .select("instructor_id", "week_start_date", "avg_completion_rate")
            .withColumn(
                "completion_rate_4w_avg",
                F.round(
                    F.avg("avg_completion_rate").over(lookback_window),
                    4
                )
            )
            # Keep only the most recent row per instructor (latest week's 4w avg)
            .withColumn("row_rank", F.row_number().over(
                Window.partitionBy("instructor_id").orderBy(F.desc("week_start_date"))
            ))
            .filter(F.col("row_rank") == 1)
            .select("instructor_id", "completion_rate_4w_avg")
        )

        print(f"✅ 4-week rolling average computed from {prior_weeks_count} prior weeks")
        four_week_avg_df.show(5)
    else:
        # Not enough history — create empty df, will be filled with current value
        four_week_avg_df = prior_snapshots.select("instructor_id") \
                                           .limit(0) \
                                           .withColumn("completion_rate_4w_avg", F.lit(None).cast(DoubleType()))
        print("ℹ  Only 1 prior week — not enough for meaningful 4w avg, using current value")

else:
    # First run — no prior table
    from pyspark.sql.types import StructType, StructField
    empty_schema = StructType([
        StructField("instructor_id",          StringType(), True),
        StructField("completion_rate_4w_avg", DoubleType(), True),
    ])
    four_week_avg_df = spark.createDataFrame([], empty_schema)
    print("ℹ  First run — no prior snapshots. completion_rate_4w_avg = current week value")

In [0]:
# ============================================================
# CELL 17 — Join 4-week avg + high risk counts
#           Assemble final Gold DataFrame with all columns
#           Add week_start_date and Gold metadata
# ============================================================

# Join 4-week avg (left join so first run still works)
instructor_with_trend_df = (
    instructor_ranked_df
    .join(four_week_avg_df, on="instructor_id", how="left")
    .join(high_risk_per_instructor, on="instructor_id", how="left")
    .fillna({
        "students_at_high_dropout_risk": 0,
    })
    # If no prior 4w data, use current week value as the avg
    .withColumn(
        "completion_rate_4w_avg",
        F.coalesce(
            F.col("completion_rate_4w_avg"),
            F.col("avg_completion_rate")
        )
    )
    # Week-over-week delta: positive = improving vs 4-week trend
    .withColumn(
        "week_over_week_delta",
        F.round(
            F.col("avg_completion_rate") - F.col("completion_rate_4w_avg"),
            4
        )
    )
)

# Assemble final column order
instructor_dashboard_df = (
    instructor_with_trend_df
    .select(
        # ── Identity ──────────────────────────────────────────
        "instructor_id",
        "instructor_name",
        # ── Temporal grain ────────────────────────────────────
        F.lit(WEEK_START_DATE).cast(DateType()).alias("week_start_date"),
        # ── Course + student counts ───────────────────────────
        "active_courses",
        "total_active_students",
        "total_dropout_count",
        # ── Performance metrics ───────────────────────────────
        "avg_completion_rate",
        "avg_quiz_pass_rate",
        "avg_engagement_score",
        # ── Risk metric ───────────────────────────────────────
        "students_at_high_dropout_risk",
        # ── Ranking ───────────────────────────────────────────
        "performance_rank",
        # ── Historical trend ──────────────────────────────────
        "completion_rate_4w_avg",
        "week_over_week_delta",
    )
    # Add Gold metadata
    .withColumn("_gold_load_ts",   F.current_timestamp())
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
)

print("── Final Instructor Dashboard DataFrame ──────────")
print(f"   Rows    : {instructor_dashboard_df.count():,}")
print(f"   Columns : {len(instructor_dashboard_df.columns)}")
print()
instructor_dashboard_df.printSchema()

In [0]:
# ============================================================
# CELL 18 — Write to gold.instructor_dashboard
#
# Write strategy:
#   - Partitioned by week_start_date (one partition per weekly run)
#   - replaceWhere for idempotent writes (re-running same week
#     replaces only that week's partition, not historical data)
#   - On first run: plain overwrite creates the table
#
# Weekly cadence: Airflow triggers this every Monday at 05:00 AM
# ============================================================

table_exists = spark.catalog.tableExists(INSTRUCTOR_DASH_TABLE)

write_options = (
    instructor_dashboard_df.write
    .format("delta")
    .partitionBy("week_start_date")
    .option("mergeSchema", "true")
)

if table_exists:
    (
        write_options
        .mode("overwrite")
        .option("replaceWhere", f"week_start_date = '{WEEK_START_DATE}'")
        .saveAsTable(INSTRUCTOR_DASH_TABLE)
    )
    print(f"✅ Written (replaceWhere) to: {INSTRUCTOR_DASH_TABLE}")
else:
    (
        write_options
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(INSTRUCTOR_DASH_TABLE)
    )
    print(f"✅ Written (first run, full overwrite) to: {INSTRUCTOR_DASH_TABLE}")

final_count = spark.table(INSTRUCTOR_DASH_TABLE).count()
print(f"   Total rows in table  : {final_count:,}")
print(f"   week_start_date      : {WEEK_START_DATE}")

In [0]:
# ============================================================
# CELL 19 — Verify gold.instructor_dashboard
# ============================================================

dash_df = spark.table(INSTRUCTOR_DASH_TABLE)

print("── instructor_dashboard ──────────────────────────")
print(f"Total rows    : {dash_df.count():,}")
print(f"Columns       : {len(dash_df.columns)}")
print()

print("── Performance rank overview ─────────────────────")
dash_df.select(
    F.min("performance_rank").alias("best_rank"),
    F.max("performance_rank").alias("worst_rank"),
    F.countDistinct("performance_rank").alias("distinct_ranks"),
    F.count("instructor_id").alias("total_instructors")
).show()

print("── Top 10 instructors (ranked) ───────────────────")
dash_df.orderBy("performance_rank").select(
    "performance_rank",
    "instructor_name",
    "active_courses",
    "total_active_students",
    "avg_completion_rate",
    "avg_quiz_pass_rate",
    "students_at_high_dropout_risk",
    "week_over_week_delta"
).show(10, truncate=False)

print("── Instructors with most at-risk students ────────")
dash_df.orderBy(F.desc("students_at_high_dropout_risk")).select(
    "instructor_name",
    "students_at_high_dropout_risk",
    "total_active_students",
    "avg_completion_rate",
    "performance_rank"
).show(5, truncate=False)

print("── Week-over-week trend distribution ────────────")
dash_df.select(
    F.when(F.col("week_over_week_delta") > 0.01, "IMPROVING")
     .when(F.col("week_over_week_delta") < -0.01, "DECLINING")
     .otherwise("STABLE")
     .alias("completion_trend")
).groupBy("completion_trend").count().show()

In [0]:
# ============================================================
# CELL 20 — All DQ checks for instructor_dashboard
#
# Check 1: week_start_date must not be NULL
# Check 2: avg_completion_rate must be in [0.0, 1.0]
# Check 3: performance_rank must be >= 1 and unique per week
# Check 4: students_at_high_dropout_risk must be >= 0
# Check 5: every active instructor from catalog must appear
# ============================================================

dash_df = spark.table(INSTRUCTOR_DASH_TABLE)
total   = dash_df.count()

print("── DQ Check 1: NULL week_start_date ──────────────")
null_week = dash_df.filter(F.col("week_start_date").isNull()).count()
print(f"   {'✅ PASSED' if null_week == 0 else f'⚠ {null_week} NULL week_start_date rows'}")
print()

print("── DQ Check 2: avg_completion_rate in [0, 1] ─────")
invalid_rate = dash_df.filter(
    (F.col("avg_completion_rate") < 0) | (F.col("avg_completion_rate") > 1)
).count()
if invalid_rate > 0:
    (
        dash_df.filter((F.col("avg_completion_rate") < 0) | (F.col("avg_completion_rate") > 1))
        .withColumn("dq_check_name", F.lit("avg_completion_rate_out_of_range"))
        .withColumn("dq_table",      F.lit(INSTRUCTOR_DASH_TABLE))
        .withColumn("dq_severity",   F.lit("HIGH"))
        .withColumn("dq_checked_at", F.current_timestamp())
        .withColumn("dq_message",    F.concat_ws(" | ",
            F.lit("avg_completion_rate out of [0,1]:"),
            F.col("avg_completion_rate").cast("string"),
            F.lit("instructor_id:"), F.col("instructor_id")
        ))
        .select("dq_check_name", "dq_table", "dq_severity", "dq_checked_at", "dq_message", "instructor_id")
        .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {invalid_rate} rows out of range → {DQ_AUDIT_TABLE}")
else:
    print("   ✅ PASSED")
print()

print("── DQ Check 3: performance_rank >= 1 ────────────")
invalid_rank = dash_df.filter(F.col("performance_rank") < 1).count()
print(f"   {'✅ PASSED' if invalid_rank == 0 else f'⚠ {invalid_rank} rows with rank < 1'}")
print()

print("── DQ Check 4: students_at_high_dropout_risk >= 0 ─")
negative_risk = dash_df.filter(F.col("students_at_high_dropout_risk") < 0).count()
print(f"   {'✅ PASSED' if negative_risk == 0 else f'⚠ {negative_risk} rows with negative risk count'}")
print()

print("── DQ Check 5: all active instructors present ────")
expected_instructors = instructor_course_map.select("instructor_id").distinct().count()
actual_instructors   = dash_df.select("instructor_id").distinct().count()
missing_instructors  = expected_instructors - actual_instructors
print(f"   Expected (active in catalog) : {expected_instructors:,}")
print(f"   Present in dashboard         : {actual_instructors:,}")
if missing_instructors > 0:
    print(f"   ⚠ {missing_instructors} instructors missing from dashboard")
else:
    print(f"   ✅ PASSED — all active instructors have a dashboard row")

In [0]:
%sql
-- ============================================================
-- CELL 21 — SQL verification of gold.instructor_dashboard
-- ============================================================

SELECT
    week_start_date,
    COUNT(*)                                           AS total_instructors,
    ROUND(AVG(active_courses), 2)                      AS avg_active_courses,
    ROUND(AVG(total_active_students), 2)               AS avg_students_per_instructor,
    ROUND(AVG(avg_completion_rate), 4)                 AS overall_avg_completion_rate,
    ROUND(AVG(avg_quiz_pass_rate), 4)                  AS overall_avg_quiz_pass_rate,
    ROUND(AVG(avg_engagement_score), 2)                AS overall_avg_engagement,
    SUM(students_at_high_dropout_risk)                 AS total_high_risk_students,
    MIN(performance_rank)                              AS top_rank,
    MAX(performance_rank)                              AS bottom_rank,
    SUM(CASE WHEN week_over_week_delta > 0.01 THEN 1 ELSE 0 END) AS improving_instructors,
    SUM(CASE WHEN week_over_week_delta < -0.01 THEN 1 ELSE 0 END) AS declining_instructors
FROM mini_project_grp6.gold.instructor_dashboard
GROUP BY week_start_date
ORDER BY week_start_date DESC;

In [0]:
# ============================================================
# CELL 22 — Task 3.2 completion summary
# ============================================================

final_df      = spark.table(INSTRUCTOR_DASH_TABLE)
final_count   = final_df.count()
top_rank_name = final_df.filter(F.col("performance_rank") == 1) \
                         .select("instructor_name").collect()[0][0]
high_risk_total = final_df.agg(
    F.sum("students_at_high_dropout_risk")
).collect()[0][0]

print("═" * 62)
print("  TASK 3.2 COMPLETE — Gold Instructor Dashboard")
print("═" * 62)
print()
print(f"  ✅ {INSTRUCTOR_DASH_TABLE}")
print(f"      Rows (instructors)        : {final_count:,}")
print(f"      week_start_date           : {WEEK_START_DATE}")
print(f"      Partition                 : week_start_date")
print(f"      Write mode                : replaceWhere(week_start_date)")
print(f"      Refresh cadence           : Weekly on Monday")
print()
print("  Metrics in output:")
print("      active_courses             — courses with at least 1 active student")
print("      total_active_students      — sum across all instructor's courses")
print("      avg_completion_rate        — AVG(completion_rate) across courses")
print("      avg_quiz_pass_rate         — satisfaction proxy (AVG pass rate)")
print("      avg_engagement_score       — AVG composite score across courses")
print("      students_at_high_dropout_risk — HIGH tier count from risk profile")
print(f"      performance_rank           — RANK() by avg_completion_rate DESC")
print(f"      completion_rate_4w_avg     — rolling {LOOKBACK_WEEKS}-week average")
print(f"      week_over_week_delta       — current vs 4w avg (+ = improving)")
print()
print(f"  Rank #1 instructor this week  : {top_rank_name}")
print(f"  Total HIGH risk students      : {high_risk_total:,}")
print()
print("  DQ checks run:")
print("      Cell 20 — NULL week_start_date")
print("      Cell 20 — avg_completion_rate in [0, 1]")
print("      Cell 20 — performance_rank >= 1")
print("      Cell 20 — students_at_high_dropout_risk >= 0")
print("      Cell 20 — all active instructors present")
print()
print("  ─────────────────────────────────────────────────────")
print("  Next: Task 3.3 → 10_gold_completion_funnel")
print("         6-stage funnel + biggest drop-off + Q1 vs Q2 cohort")
print("═" * 62)